In [5]:
import os
import pandas as pd

raw_path = "../data/raw"
files = os.listdir(raw_path)
print(files)

['AQI Bangladesh.csv', 'cities.csv']


In [8]:
aqi = pd.read_csv(os.path.join(raw_path, "AQI Bangladesh.csv"))
cities = pd.read_csv(os.path.join(raw_path, "cities.csv"))

print("AQI:", aqi.shape)
print("Cities:", cities.shape)

AQI: (1048551, 13)
Cities: (103, 4)


In [9]:
print(aqi.columns.tolist())
aqi.head()

['city_id', 'city_name', 'lat', 'lon', 'datetime', 'pm10', 'pm2_5', 'carbon_monoxide', 'carbon_dioxide', 'nitrogen_dioxide', 'sulphur_dioxide', 'ozone', 'aqi']


,city_id,city_name,lat,lon,datetime,pm10,pm2_5,carbon_monoxide,carbon_dioxide,nitrogen_dioxide,sulphur_dioxide,ozone,aqi
0,7701354,Azimpur,23.7298,90.3854,2022-08-05T00:00,25.0,16.7,252.0,NaN,18.6,5.2,13.0,50.0
1,7701354,Azimpur,23.7298,90.3854,2022-08-05T01:00,18.5,12.3,249.0,NaN,18.5,5.7,19.0,50.0
2,7701354,Azimpur,23.7298,90.3854,2022-08-05T02:00,16.1,10.9,244.0,NaN,18.4,6.5,27.0,51.0
3,7701354,Azimpur,23.7298,90.3854,2022-08-05T03:00,15.4,10.3,233.0,NaN,17.2,6.9,39.0,51.0
4,7701354,Azimpur,23.7298,90.3854,2022-08-05T04:00,16.8,11.3,217.0,NaN,14.8,6.2,51.0,51.0


In [10]:
print(cities.columns.tolist())
cities.head()

['ID', 'Name', 'Latitude', 'Longitude']


,ID,Name,Latitude,Longitude
0,1,Rangpur,25.74664,89.25166
1,2,Sherpur,25.01881,90.01751
2,3,Dinājpur,25.62745,88.63779
3,4,Lākshām,23.24018,91.12143
4,5,Comilla,23.46186,91.18503


In [11]:
aqi.info()
print(aqi.isnull().sum())

<class 'pandas.DataFrame'>
RangeIndex: 1048551 entries, 0 to 1048550
Data columns (total 13 columns):
 #   Column            Non-Null Count    Dtype  
---  ------            --------------    -----  
 0   city_id           1048551 non-null  int64  
 1   city_name         1048551 non-null  str    
 2   lat               1048551 non-null  float64
 3   lon               1048551 non-null  float64
 4   datetime          1048551 non-null  str    
 5   pm10              1048551 non-null  float64
 6   pm2_5             1048551 non-null  float64
 7   carbon_monoxide   1048551 non-null  float64
 8   carbon_dioxide    274224 non-null   float64
 9   nitrogen_dioxide  1048551 non-null  float64
 10  sulphur_dioxide   1048551 non-null  float64
 11  ozone             1048551 non-null  float64
 12  aqi               1047855 non-null  float64
dtypes: float64(10), int64(1), str(2)
memory usage: 128.6 MB
city_id                  0
city_name                0
lat                      0
lon                  

In [12]:
aqi["datetime"] = pd.to_datetime(aqi["datetime"])

summary = (
    aqi.groupby("city_name")
       .agg(rows=("aqi", "size"),
            start=("datetime", "min"),
            end=("datetime", "max"),
            lat=("lat", "first"),
            lon=("lon", "first"))
       .sort_values("rows", ascending=False)
)
summary.head(20)

,rows,start,end,lat,lon
city_name,,,,,
Dhaka,227016,2000-01-01,2025-11-23 23:00:00,23.71040,90.40744
Badarganj,28992,2022-08-04,2025-11-23 23:00:00,25.67419,89.05377
Bera,28992,2022-08-04,2025-11-23 23:00:00,24.07821,89.63262
Barisāl,28992,2022-08-04,2025-11-23 23:00:00,22.70497,90.37013
Bhairab Bāzār,28992,2022-08-04,2025-11-23 23:00:00,24.05240,90.97640
Baniachang,28992,2022-08-04,2025-11-23 23:00:00,24.51863,91.35787
Bogra,28992,2022-08-04,2025-11-23 23:00:00,24.85098,89.37108
Bhola,28992,2022-08-04,2025-11-23 23:00:00,22.68759,90.64403
Bhāndāria,28992,2022-08-04,2025-11-23 23:00:00,22.48898,90.06273


In [13]:
dhaka = summary[
    summary["lat"].between(23.65, 23.95) &
    summary["lon"].between(90.30, 90.55)
]
print(len(dhaka))
dhaka

2


,rows,start,end,lat,lon
city_name,,,,,
Dhaka,227016,2000-01-01,2025-11-23 23:00:00,23.7104,90.40744
Azimpur,28968,2022-08-05,2025-11-23 23:00:00,23.7298,90.38540


In [14]:
print(aqi["datetime"].min(), aqi["datetime"].max())
print(aqi["datetime"].dt.year.value_counts().sort_index())

2000-01-01 00:00:00 2025-11-23 23:00:00
datetime
2000      8784
2001      8760
2002      8760
2003      8760
2004      8784
2005      8760
2006      8760
2007      8760
2008      8784
2009      8760
2010      8760
2011      8760
2012      8784
2013      8760
2014      8760
2015      8760
2016      8784
2017      8760
2018      8760
2019      8760
2020      8784
2021      8760
2022    113136
2023    260223
2024    254736
2025    227592
Name: count, dtype: int64


In [15]:
dhaka = aqi[aqi["city_name"] == "Dhaka"].copy()
dhaka = dhaka.sort_values("datetime").reset_index(drop=True)
print(dhaka.shape)
dhaka.head()

(227016, 13)


,city_id,city_name,lat,lon,datetime,pm10,pm2_5,carbon_monoxide,carbon_dioxide,nitrogen_dioxide,sulphur_dioxide,ozone,aqi
0,1185241,Dhaka,23.7104,90.40744,2000-01-01 00:00:00,93.224101,44.200712,1.049477,NaN,15.391414,28.219932,20.586862,121.861086
1,1185241,Dhaka,23.7104,90.40744,2000-01-01 01:00:00,78.893066,35.212704,0.943859,NaN,26.824534,24.385188,22.429471,99.598077
2,1185241,Dhaka,23.7104,90.40744,2000-01-01 02:00:00,118.693959,48.575428,0.887131,NaN,33.403493,28.951413,37.838094,132.852834
3,1185241,Dhaka,23.7104,90.40744,2000-01-01 03:00:00,86.427166,64.208516,0.793261,NaN,12.137395,26.950990,26.747684,154.588259
4,1185241,Dhaka,23.7104,90.40744,2000-01-01 04:00:00,70.193382,41.112699,1.059887,NaN,25.313206,20.741976,33.192759,114.102260


In [16]:
print("Duplicate timestamps:", dhaka["datetime"].duplicated().sum())

full_range = pd.date_range(dhaka["datetime"].min(), dhaka["datetime"].max(), freq="h")
print("Missing hours:", len(full_range) - len(dhaka))

Duplicate timestamps: 0
Missing hours: 0


In [17]:
print(dhaka.isnull().sum())

city_id                  0
city_name                0
lat                      0
lon                      0
datetime                 0
pm10                     0
pm2_5                    0
carbon_monoxide          0
carbon_dioxide      217560
nitrogen_dioxide         0
sulphur_dioxide          0
ozone                    0
aqi                     24
dtype: int64


In [18]:
# carbon_dioxide-এ বেশিরভাগ মান ফাঁকা, তাই বাদ। city_id, lat, lon আর লাগবে না।
cols = ["datetime", "pm2_5", "pm10", "nitrogen_dioxide",
        "sulphur_dioxide", "ozone", "carbon_monoxide", "aqi"]
dhaka = dhaka[cols]

In [19]:
dhaka.describe().T

,count,mean,min,25%,50%,75%,max,std
datetime,227016,2012-12-12 11:29:59.999999,2000-01-01 00:00:00,2006-06-22 17:45:00,2012-12-12 11:30:00,2019-06-04 05:15:00,2025-11-23 23:00:00,NaN
pm2_5,227016.0,89.686352,0.4,51.606915,81.688179,119.807446,290.6,50.98565
pm10,227016.0,150.039419,0.6,83.4,136.958251,204.10401,415.4,88.256501
nitrogen_dioxide,227016.0,33.191176,0.0,23.330383,32.084546,41.262418,178.0,15.266497
sulphur_dioxide,227016.0,21.184527,0.2,15.768988,20.836639,26.220703,60.2,7.995303
ozone,227016.0,52.51158,0.0,36.649487,48.761429,63.0,298.0,26.106073
carbon_monoxide,227016.0,76.248701,0.1,1.27165,1.662784,2.167506,4313.0,259.707385
aqi,226992.0,159.674116,13.0,141.986991,163.808256,183.743897,299.5996,47.448165


In [20]:
(dhaka["pm2_5"] < 0).sum(), (dhaka["pm2_5"] > 500).sum()

(np.int64(0), np.int64(0))

In [21]:
dhaka["year"] = dhaka["datetime"].dt.year
dhaka["month"] = dhaka["datetime"].dt.month
dhaka["hour"] = dhaka["datetime"].dt.hour
dhaka["day_of_week"] = dhaka["datetime"].dt.dayofweek

def season(m):
    if m in (12, 1, 2): return "Winter"
    if m in (3, 4, 5): return "Summer"
    if m in (6, 7, 8, 9): return "Monsoon"
    return "Post-monsoon"

dhaka["season"] = dhaka["month"].map(season)
dhaka.head()

,datetime,pm2_5,pm10,nitrogen_dioxide,sulphur_dioxide,ozone,carbon_monoxide,aqi,year,month,hour,day_of_week,season
0,2000-01-01 00:00:00,44.200712,93.224101,15.391414,28.219932,20.586862,1.049477,121.861086,2000,1,0,5,Winter
1,2000-01-01 01:00:00,35.212704,78.893066,26.824534,24.385188,22.429471,0.943859,99.598077,2000,1,1,5,Winter
2,2000-01-01 02:00:00,48.575428,118.693959,33.403493,28.951413,37.838094,0.887131,132.852834,2000,1,2,5,Winter
3,2000-01-01 03:00:00,64.208516,86.427166,12.137395,26.950990,26.747684,0.793261,154.588259,2000,1,3,5,Winter
4,2000-01-01 04:00:00,41.112699,70.193382,25.313206,20.741976,33.192759,1.059887,114.102260,2000,1,4,5,Winter


In [22]:
dhaka.to_csv("../data/processed/dhaka_clean.csv", index=False)
print("Saved:", dhaka.shape)

Saved: (227016, 13)


In [23]:
dhaka["year"] = dhaka["datetime"].dt.year
dhaka.groupby("year")["carbon_monoxide"].agg(["mean", "median", "max"]).round(2)

,mean,median,max
year,,,
2000,1.20,1.20,2.50
2001,1.23,1.24,2.57
2002,1.27,1.27,2.60
2003,1.31,1.31,2.57
2004,1.34,1.34,2.72
2005,1.38,1.37,2.75
2006,1.41,1.41,2.82
2007,1.46,1.46,2.92
2008,1.49,1.49,2.81


In [24]:
dhaka.groupby("year")[["pm2_5", "pm10", "aqi"]].mean().round(1)

,pm2_5,pm10,aqi
year,,,
2000,39.6,68.2,107.2
2001,43.7,74.7,114.9
2002,47.6,80.9,121.8
2003,51.9,88.6,128.1
2004,56.8,96.5,135.2
2005,61.4,104.6,140.9
2006,66.4,113.1,146.2
2007,71.5,121.3,151.5
2008,76.6,130.1,155.9


In [25]:
az = aqi[aqi["city_name"] == "Azimpur"].copy()
az["datetime"] = pd.to_datetime(az["datetime"])

both = dhaka[["datetime", "pm2_5"]].merge(
    az[["datetime", "pm2_5"]], on="datetime", suffixes=("_dhaka", "_azimpur")
)
print(both.shape)
print(both[["pm2_5_dhaka", "pm2_5_azimpur"]].describe().round(1))
print("Correlation:", both["pm2_5_dhaka"].corr(both["pm2_5_azimpur"]).round(3))

(28968, 3)
       pm2_5_dhaka  pm2_5_azimpur
count      28968.0        28968.0
mean          48.7           48.7
std           37.0           37.0
min            0.4            0.4
25%           19.7           19.7
50%           39.7           39.7
75%           68.2           68.2
max          290.6          290.6
Correlation: 1.0


In [26]:
dhaka[["pm2_5", "aqi"]].sort_values("pm2_5").iloc[::25000]

,pm2_5,aqi
201210,0.400000,14.000000
214337,32.200000,62.000000
11538,48.013981,131.442164
67361,61.118227,152.960078
212490,74.300000,152.000000
160393,88.296840,167.279684
99760,104.100309,175.606064
178710,124.146216,186.167658
160813,153.262407,202.765173
166118,250.000000,299.599600


In [27]:
cut = dhaka[dhaka["carbon_monoxide"] > 50]["datetime"].min()
print("Cutover:", cut)

Cutover: 2022-08-04 00:00:00


In [28]:
pre = dhaka[(dhaka["datetime"] < cut) & dhaka["year"].between(2015, 2019)]
post = dhaka[dhaka["datetime"] >= cut]

compare = pd.DataFrame({
    "pre_2015_2019": pre.groupby(pre["datetime"].dt.month)["pm2_5"].mean().round(1),
    "post_cutover": post.groupby(post["datetime"].dt.month)["pm2_5"].mean().round(1),
})
compare

,pre_2015_2019,post_cutover
datetime,,
1,175.6,98.2
2,175.1,80.8
3,117.2,58.1
4,117.0,46.3
5,117.2,35.7
6,116.8,26.2
7,117.5,14.4
8,117.2,20.5
9,117.4,26.0


In [29]:
d = dhaka[dhaka["datetime"] >= cut].copy().reset_index(drop=True)

d["month"] = d["datetime"].dt.month
d["hour"] = d["datetime"].dt.hour
d["day_of_week"] = d["datetime"].dt.dayofweek

def season(m):
    if m in (12, 1, 2): return "Winter"
    if m in (3, 4, 5): return "Summer"
    if m in (6, 7, 8, 9): return "Monsoon"
    return "Post-monsoon"

d["season"] = d["month"].map(season)
d = d.drop(columns=["aqi", "carbon_dioxide", "city_id", "city_name", "lat", "lon"], errors="ignore")

print(d.shape, d["datetime"].min(), d["datetime"].max())
d.head()

(28992, 12) 2022-08-04 00:00:00 2025-11-23 23:00:00


,datetime,pm2_5,pm10,nitrogen_dioxide,sulphur_dioxide,ozone,carbon_monoxide,year,month,hour,day_of_week,season
0,2022-08-04 00:00:00,13.1,19.0,17.5,2.5,17.0,243.0,2022,8,0,3,Monsoon
1,2022-08-04 01:00:00,9.2,13.5,16.8,2.9,18.0,234.0,2022,8,1,3,Monsoon
2,2022-08-04 02:00:00,9.9,14.5,15.9,3.4,20.0,221.0,2022,8,2,3,Monsoon
3,2022-08-04 03:00:00,8.5,12.7,14.1,3.8,26.0,202.0,2022,8,3,3,Monsoon
4,2022-08-04 04:00:00,8.4,12.5,11.8,3.7,34.0,186.0,2022,8,4,3,Monsoon


In [30]:
def pm25_aqi(c):
    c = np.floor(c * 10) / 10
    bp = [(0.0, 9.0, 0, 50), (9.1, 35.4, 51, 100), (35.5, 55.4, 101, 150),
          (55.5, 125.4, 151, 200), (125.5, 225.4, 201, 300), (225.5, 325.4, 301, 500)]
    for lo, hi, ilo, ihi in bp:
        if lo <= c <= hi:
            return (ihi - ilo) / (hi - lo) * (c - lo) + ilo
    return np.nan

d["aqi_calc"] = d["pm2_5"].apply(pm25_aqi)
d[["pm2_5", "aqi_calc"]].describe().round(1)

,pm2_5,aqi_calc
count,28992.0,28992.0
mean,48.6,116.8
std,37.0,52.3
min,0.4,2.2
25%,19.7,70.7
50%,39.6,111.1
75%,68.2,159.9
max,290.6,430.7


In [31]:
d.to_csv("../data/processed/dhaka_clean.csv", index=False)
print("Saved:", d.shape)

Saved: (28992, 13)
